In [1]:
import pinecone
from pinecone import Pinecone, ServerlessSpec
import configPinecone

In [2]:
pc = Pinecone(api_key = configPinecone.pineconeDefaultAPIKey, environment = configPinecone.pineconeEnv)

In [3]:
pc.list_indexes()

[
    {
        "name": "my-index",
        "metric": "cosine",
        "host": "my-index-cwx78x5.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 3,
        "deletion_protection": "disabled",
        "tags": null
    }
]

In [5]:
pc.delete_index("my-index")
pc.create_index(
    name = "my-index",
    dimension = 3,
    metric = "cosine",
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-cwx78x5.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 3,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
  

In [6]:
index = pc.Index(name = "my-index")

In [7]:
index.upsert([
    ("Dog", [4., 0., 1.]),
    ("Cat", [4., 0., 1.]),
    ("Elephant", [4., 0., 1.]),
    ("Chicken", [2., 2., 1.]),
    ("Mantis", [6., 2., 3.])
])

UpsertResponse(upserted_count=5, _response_info={'raw_headers': {'date': 'Tue, 03 Mar 2026 09:27:48 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '87', 'x-pinecone-request-latency-ms': '294', 'x-envoy-upstream-service-time': '323', 'x-pinecone-response-duration-ms': '324', 'grpc-status': '0', 'server': 'envoy'}})

In [8]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [9]:
fw = load_dataset("HuggingFaceFw/fineweb", name = "sample-10BT", split = "train", streaming = True)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [10]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [11]:
fw.features

{'text': Value(dtype='string', id=None),
 'id': Value(dtype='string', id=None),
 'dump': Value(dtype='string', id=None),
 'url': Value(dtype='string', id=None),
 'date': Value(dtype='string', id=None),
 'file_path': Value(dtype='string', id=None),
 'language': Value(dtype='string', id=None),
 'language_score': Value(dtype='float64', id=None),
 'token_count': Value(dtype='int64', id=None)}

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\anaconda3\envs\llm\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
pc.create_index(name="embedding-index",
               dimension = model.get_sentence_embedding_dimension(),
                metric = "cosine",
                spec = ServerlessSpec(
                    cloud = "aws",
                    region = "us-east-1"
                )
               )

{
    "name": "embedding-index",
    "metric": "cosine",
    "host": "embedding-index-cwx78x5.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version

In [15]:
index = pc.Index(name = "embedding-index")

In [16]:
subsetSize = 10_000 # num of items to process
#iterate over the dataset and prepare data for upserting
vectorsToUpsert = []
for i, item in enumerate(fw):
    if i >= subsetSize:
        break
    
    text = item["text"]
    uniqueId = str(item["id"])
    metadata = {"language": item["language"]}
    
    embedding = model.encode(text, show_progress_bar = False).tolist()

    vectorsToUpsert.append((uniqueId, embedding, metadata))

# upsert data to pinecone in batches
batchSize = 1000
for i in range(0, len(vectorsToUpsert), batchSize):
    batch = vectorsToUpsert[i: i + batchSize]
    index.upsert(vectors = batch)

print("Subset of data upserted to the Pinecone index")

Subset of data upserted to the Pinecone index


In [17]:
# took 15 mins to execute